## 8-Puzzle Problem using Searching Algorithms


The 8-puzzle is a sliding tile game on a \($3 \times 3$) grid solved optimally using the A* Search Algorithm through the evaluation function

$$f(n) = g(n) + h(n)$$

Overview of the 8-Puzzle and A*:

- **The Grid:** A $3 \times 3$ frame containing 8 numbered tiles (1 through 8) and one blank/empty space.

- **The Objective:** Rearrange the tiles from a given start state to reach a goal configuration by sliding tiles horizontally or vertically into the empty space.

- **State Space:** Contains $9! / 2 = 181,440$ solvable configurations with an average optimal solution path length of about 22 moves.

Core Components of A* Search:

- **$g(n)$ (Path Cost):** The actual number of moves taken from the start state to reach the current node $n$.

- **$h(n)$ (Heuristic Function):** The estimated cost from the current state to the goal.

Common heuristics include:

- **Misplaced (Missing) Tiles:** Count of tiles that are not in their correct goal positions.

- **Manhattan Distance:** The sum of the horizontal and vertical distances that each tile is away from its goal position (dominates misplaced tiles and yields superior efficiency).

**Evaluation Function: **Total estimated cost of the cheapest path through $n$, defined as $f(n) = g(n) + h(n)$.

Source Code:

https://github.com/memoodm/AI-8Puzzle-SearchAlgorithm/blob/master/README.md

https://github.com/mahanthmukesh/8-puzzle-using-A-algorithm/blob/master/code/node.py

#### I. Node and Representation Functions

In this section, we define the basic dictionary structure used to represent a state (node) in the search space. Instead of using a heavy class object, a simple dictionary is used containing:
- `'child'`: Current configuration of the board.
- `'parent'`: Reference to the parent state (used for backtracking and reconstructing the solution path).
- `'gn'`: The cost to reach this node from the initial state.
- `'hn'`: The estimated heuristic distance from this state to the goal.

We also define clean, functional getter and setter utilities (`get_parent`, `get_hn`, `update_gn`, etc.) to manipulate these state dictionaries procedurally.

In [29]:
import numpy as np

# --- Node related functions ---
def create_node(s):
    """Creates a dictionary representation of a node."""
    return {
        'child': s,
        'parent': None,
        'gn': 0,
        'hn': 0,
    }

def get_parent(node):
    """Returns the parent node."""
    return node['parent']

def get_hn(node):
    """Returns the heuristic cost (h(n))."""
    return node['hn']

def get_fn(node):
    """Returns the total cost (f(n) = g(n) + h(n))."""
    return node['gn'] + node['hn']

def get_gn(node):
    """Returns the cost from start to current node (g(n))."""
    return node['gn']

def get_current_state(node):
    """Returns the current puzzle state (child)."""
    return node['child']

def update_gn(node, gn):
    """Updates the g(n) value of a node."""
    node['gn'] = gn

def update_hn(node, hn):
    """Updates the h(n) value of a node."""
    node['hn'] = hn

def update_parent(node, parent):
    """Updates the parent of a node."""
    node['parent'] = parent

### II. Distance and Heuristic Evaluation
This block contains the core heuristic calculations used by the A* algorithm to guide its path efficiently:
- **Heuristic 1 (Misplaced Tiles):** Scans the board configuration and counts the number of non-blank tiles (excluding `0`) that are currently not in their correct goal configurations.
- **Heuristic 2 (Manhattan Distance):** Reshapes the board into a $3 \times 3$ grid and calculates the absolute vertical and horizontal distances that every tile from 1 to 8 needs to travel to reach its correct position.

In [30]:
# --- Distance Calculation Function ---
def calculate_distance(arr, goal, heuristic):
    """
    Calculates the heuristic distance between two states.
    Heuristic 1: Misplaced Tiles (excluding the blank tile)
    Heuristic 2: Manhattan Distance (excluding the blank tile)
    """
    distance_val = 0
    if heuristic == 1: # Misplaced Tiles
        for i in range(len(arr)): # Iterate through all positions
            # Only count non-blank tiles that are not in their goal position
            if arr[i] != 0 and arr[i] != goal[i]:
                distance_val += 1
        return distance_val
    elif heuristic == 2: # Manhattan Distance
        arr_np = np.asarray(arr).reshape(3,3)
        goal_np = np.asarray(goal).reshape(3,3)

        # Iterate for tiles 1 through 8 (excluding the blank tile 0)
        for i in range(1, 9):
            # Find coordinates of tile 'i' in current and goal states
            a_coords = np.argwhere(arr_np == i)[0]
            x_coords = np.argwhere(goal_np == i)[0]
            # Add Manhattan distance for this tile
            distance_val += abs(a_coords[0] - x_coords[0]) + abs(a_coords[1] - x_coords[1])
        return distance_val
    return 0 # Default return if heuristic is not 1 or 2

### III. State Expansion logic
This section defines `expand_node`, which generates valid neighboring states:
1. It marks the current state as explored.
2. It locates the empty slot (index of `0`).
3. It attempts to slide tiles into the empty space in four directions (Left, Right, Up, Down).
4. For each potential move, it constructs a new state configuration and ensures it hasn't already been explored before inserting it into the `fringe` (priority queue).

In [31]:
# --- Node Expansion Function ---
def expand_node(fringe, explored_nodes, current_node, goal_node, zero_index, g, count, heuristic):
    """
    Expands the current node, generating new possible states, and adds them to the fringe
    if they haven't been explored.
    """
    # Create a list of current states of explored nodes for quick lookup
    explored_states = [get_current_state(item) for item in explored_nodes]
    # Add the current node to the list of explored nodes
    explored_nodes.append(current_node)

    current_state_array = np.asarray(get_current_state(current_node))

    # Helper function to generate and add a new node if not in explored_states
    def _add_new_node_if_unexplored(state_array_new, current_node_parent, fringe_list,
                                    explored_states_list, current_g, goal_state, heuristic_choice, current_count):
        new_state_list = list(state_array_new) # Convert array to list for comparison and node creation
        if new_state_list not in explored_states_list:
            new_node = create_node(new_state_list)
            update_gn(new_node, current_g)
            distance = calculate_distance(new_state_list, goal_state, heuristic_choice)
            update_hn(new_node, distance)
            update_parent(new_node, current_node_parent)
            fringe_list.append(new_node)
            return current_count + 1 # Increment count if a new node is added
        return current_count # No increment if already explored

    # Possible moves for the blank tile (0)
    # Move left
    if zero_index % 3 != 0: # not in the first column
        node_copy_array = current_state_array.copy()
        # Swap blank with the tile to its left
        node_copy_array[zero_index], node_copy_array[zero_index - 1] = node_copy_array[zero_index - 1], node_copy_array[zero_index]
        count = _add_new_node_if_unexplored(node_copy_array, current_node, fringe, explored_states, g, goal_node, heuristic, count)

    # Move right
    if (zero_index + 1) % 3 != 0: # not in the last column
        node_copy_array = current_state_array.copy()
        # Swap blank with the tile to its right
        node_copy_array[zero_index], node_copy_array[zero_index + 1] = node_copy_array[zero_index + 1], node_copy_array[zero_index]
        count = _add_new_node_if_unexplored(node_copy_array, current_node, fringe, explored_states, g, goal_node, heuristic, count)

    # Move up
    if zero_index - 3 >= 0: # not in the first row
        node_copy_array = current_state_array.copy()
        # Swap blank with the tile above it
        node_copy_array[zero_index], node_copy_array[zero_index - 3] = node_copy_array[zero_index - 3], node_copy_array[zero_index]
        count = _add_new_node_if_unexplored(node_copy_array, current_node, fringe, explored_states, g, goal_node, heuristic, count)

    # Move down
    if zero_index + 3 < 9: # not in the last row
        node_copy_array = current_state_array.copy()
        # Swap blank with the tile below it
        node_copy_array[zero_index], node_copy_array[zero_index + 3] = node_copy_array[zero_index + 3], node_copy_array[zero_index]
        count = _add_new_node_if_unexplored(node_copy_array, current_node, fringe, explored_states, g, goal_node, heuristic, count)

    return count

### IV. Search Helpers and Path Backtracking
These procedural helper functions manage the search lifecycle:
- `find_least_fn`: Finds the index of the node in the fringe with the lowest $f(n)$ value ($g(n) + h(n)$) to determine the next state to explore.
- `print_puzzle_state`: Formats the $1 \times 9$ list representation into a traditional $3 \times 3$ visual grid.
- `handle_goal_reached`: Reconstructs the complete optimal sequence of moves by working backward from the goal node through the chain of parent pointers.

In [32]:
# --- Puzzle Solver Helper Functions ---
def find_least_fn(fringe):
    """
    Finds the index of the node with the minimum f(n) value in the fringe.
    Returns -1 if fringe is empty.
    """
    if not fringe:
        return -1
    fn_values = [get_fn(item) for item in fringe]
    minimum_fn = min(fn_values)
    minimum_fn_index = fn_values.index(minimum_fn)
    return minimum_fn_index

def print_puzzle_state(node):
    """Prints the puzzle state in a 3x3 grid format along with g(n), h(n), f(n)."""
    current_state = get_current_state(node)
    print(f"g(n) = {get_gn(node)}, h(n) = {get_hn(node)}, f(n) = {get_fn(node)}\n")
    print(f"{current_state[0]} | {current_state[1]} | {current_state[2]}")
    print("--------------")
    print(f"{current_state[3]} | {current_state[4]} | {current_state[5]}")
    print("--------------")
    print(f"{current_state[6]} | {current_state[7]} | {current_state[8]}")
    print("----------------------------------------------------------\n")

def handle_goal_reached(explored_nodes, count):
    """
    Prints the path from the initial state to the goal state,
    and displays statistics about the search.
    """
    nodes_expanded = len(explored_nodes) - 1 # Assuming the initial node is in explored_nodes

    path_to_goal = []
    current = explored_nodes[-1] # The last node in explored_nodes is the goal node

    # Reconstruct the path by backtracking through parent pointers
    while current is not None:
        path_to_goal.append(current)
        current = get_parent(current)

    path_to_goal.reverse() # Reverse to get path from start to goal

    print("Path to Goal:")
    for node_in_path in path_to_goal:
        print_puzzle_state(node_in_path)

    print("Goal Reached \n")
    print(f"The number of nodes expanded: {nodes_expanded}\n")
    print(f"The number of nodes generated: {count}\n")
    print(f"Path Cost: {len(path_to_goal) - 1}\n") # Path cost is the number of moves

### V. Solving the Puzzle

This section contains the main workflow designed to solve the 8-puzzle programmatically:
1. **Initialization:** Defines the target `goal_state` and sets up the starting layout `initial_state` using the requested tiles sequence `[8, 6, 4, 2, 1, 3, 5, 7, 0]`.
2. **Heuristic Selection:** Configures the evaluation process using **Manhattan Distance** (Heuristic 2) which generally operates much faster and generates fewer node branches than the Misplaced Tiles heuristic.
3. **Search Activation:** Calls the procedural `solve_8_puzzle` function to begin exploring and searching through state configurations.

#### ⚠️ Warning: The Infinite Loop Risk of Solvability
If you uncomment the randomizing logic (`random.sample(possible_tiles, len(possible_tiles))`) to generate initial configurations, you run a major risk of generating an **unsolvable board state**.

- **The Rule of Solvability:** In the sliding 8-puzzle, a state can only reach the goal state if the number of **inversions** (pairs of tiles that are out of numerical order) has the same parity (even or odd) as the goal state. Specifically, for a standard $3 \times 3$ grid, a state is solvable if and only if its number of inversions is **even**.
- **Infinite Search Loops:** Exactly half ($50\%$) of all possible random tile permutations are mathematical anomalies that can never reach the goal configuration. Attempting to solve one of these unsolvable states with A* Search will cause the program to generate and expand every single reachable state in the state space ($181,440$ nodes), consuming excessive memory and resulting in an infinite execution loop or crash within the cell without finding a solution.

In [33]:
import random

def solve_8_puzzle(initial_state, goal_state, heuristic_choice):
    """Solves the 8-puzzle problem using A* search."""

    # Initialize the first node
    initial_node = create_node(initial_state)
    update_hn(initial_node, calculate_distance(initial_state, goal_state, heuristic_choice))

    fringe = [initial_node]
    explored_nodes = []
    count = 1 # Number of nodes generated

    while fringe:
        # Find the node with the least f(n) in the fringe
        min_fn_index = find_least_fn(fringe)
        if min_fn_index == -1: # Fringe is empty, no solution
            print("Fringe is empty, no solution found.")
            return

        current_node = fringe.pop(min_fn_index)
        current_state = get_current_state(current_node)

        # Check if the current state is the goal state
        if current_state == goal_state:
            handle_goal_reached(explored_nodes + [current_node], count)
            return

        # Get the index of the blank tile (0)
        zero_index = current_state.index(0)

        # Expand the current node and update count
        g_cost = get_gn(current_node) + 1 # g(n) of next states
        count = expand_node(fringe, explored_nodes, current_node, goal_state, zero_index, g_cost, count, heuristic_choice)

    print("No solution found.")

# --- Main execution ---
if __name__ == '__main__':
    # Define the goal state for the 8-puzzle
    goal_state = [1, 2, 3, 4, 5, 6, 7, 8, 0]

    # Define the initial tiles given by the user
    initial_state = [8, 6, 4, 2, 1, 3, 5, 7, 0]

    # Randomize the initial state (ensure it's a valid permutation)
    # It's important to make sure the randomized state is solvable, though
    # this simple random.sample doesn't guarantee solvability.
    # initial_state = random.sample(possible_tiles, len(possible_tiles))

    print("Initial State:", initial_state)
    print("Goal State:", goal_state)

    # Choose heuristic: 1 for Misplaced Tiles, 2 for Manhattan Distance
    # Manhattan distance generally performs better for 8-puzzle
    heuristic_choice = 2
    print("Using Heuristic:", "Manhattan Distance" if heuristic_choice == 2 else "Misplaced Tiles")

    solve_8_puzzle(initial_state, goal_state, heuristic_choice)


Initial State: [8, 6, 4, 2, 1, 3, 5, 7, 0]
Goal State: [1, 2, 3, 4, 5, 6, 7, 8, 0]
Using Heuristic: Manhattan Distance
Path to Goal:
g(n) = 0, h(n) = 16, f(n) = 16

8 | 6 | 4
--------------
2 | 1 | 3
--------------
5 | 7 | 0
----------------------------------------------------------

g(n) = 1, h(n) = 17, f(n) = 18

8 | 6 | 4
--------------
2 | 1 | 0
--------------
5 | 7 | 3
----------------------------------------------------------

g(n) = 2, h(n) = 18, f(n) = 20

8 | 6 | 4
--------------
2 | 0 | 1
--------------
5 | 7 | 3
----------------------------------------------------------

g(n) = 3, h(n) = 17, f(n) = 20

8 | 0 | 4
--------------
2 | 6 | 1
--------------
5 | 7 | 3
----------------------------------------------------------

g(n) = 4, h(n) = 16, f(n) = 20

0 | 8 | 4
--------------
2 | 6 | 1
--------------
5 | 7 | 3
----------------------------------------------------------

g(n) = 5, h(n) = 15, f(n) = 20

2 | 8 | 4
--------------
0 | 6 | 1
--------------
5 | 7 | 3
---------------

### VI. Playback and Real-time Heuristic Visualization
This final section sets up the live visualization engine:
- `get_individual_heuristics`: Calculates the exact heuristic contribution (Manhattan or Misplaced) for each distinct tile individually.
- `display_puzzle_frame`: Prints a double-column view containing the visual board representation alongside a complete table breaking down the distance calculation for every numbered tile.
- `animate_solution`: Loops over the solution sequence, periodically clearing the notebook outputs and printing the next board state frame every 3 seconds to show the step-by-step solver progression.

In [34]:
import time
import numpy as np
from IPython.display import clear_output

def get_solution_path(explored_nodes):
    """Reconstructs the solution path from the start node to the goal node."""
    path = []
    current = explored_nodes[-1] # Goal node
    while current is not None:
        path.append(current)
        current = get_parent(current)
    path.reverse()
    return path

def get_individual_heuristics(arr, goal, heuristic):
    """Calculates individual heuristic distances for each tile from 1 to 8."""
    tile_heuristics = {}
    if heuristic == 1: # Misplaced Tiles
        for tile in range(1, 9):
            try:
                idx_arr = arr.index(tile)
                idx_goal = goal.index(tile)
                tile_heuristics[tile] = 1 if idx_arr != idx_goal else 0
            except ValueError:
                tile_heuristics[tile] = 0
    elif heuristic == 2: # Manhattan Distance
        arr_np = np.asarray(arr).reshape(3,3)
        goal_np = np.asarray(goal).reshape(3,3)
        for tile in range(1, 9):
            try:
                a_coords = np.argwhere(arr_np == tile)[0]
                x_coords = np.argwhere(goal_np == tile)[0]
                tile_heuristics[tile] = abs(a_coords[0] - x_coords[0]) + abs(a_coords[1] - x_coords[1])
            except IndexError:
                tile_heuristics[tile] = 0
    return tile_heuristics

def display_puzzle_frame(node, step_num, total_steps, goal_state, heuristic_choice):
    """Renders a frame of the puzzle state with a breakdown table of heuristics for each tile."""
    state = get_current_state(node)
    g = get_gn(node)
    h = get_hn(node)
    f = get_fn(node)

    tile_hs = get_individual_heuristics(state, goal_state, heuristic_choice)

    print("==========================================================")
    print(f"                   PLAYBACK FRAME: {step_num} / {total_steps}")
    print("==========================================================")
    print(f"  g(n) [Path Cost]:       {g}")
    print(f"  h(n) [Heuristic Value]: {h}")
    print(f"  f(n) [Total Cost]:      {f}")
    print("----------------------------------------------------------")
    print(f"        BOARD STATE                  TILE HEURISTIC TABLE  ")
    print(f"        {state[0]} | {state[1]} | {state[2]}                  Tile 1 distance: {tile_hs.get(1, 0)}")
    print(f"       -----------                   Tile 2 distance: {tile_hs.get(2, 0)}")
    print(f"        {state[3]} | {state[4]} | {state[5]}                  Tile 3 distance: {tile_hs.get(3, 0)}")
    print(f"       -----------                   Tile 4 distance: {tile_hs.get(4, 0)}")
    print(f"        {state[6]} | {state[7]} | {state[8]}                  Tile 5 distance: {tile_hs.get(5, 0)}")
    print(f"                                     Tile 6 distance: {tile_hs.get(6, 0)}")
    print(f"                                     Tile 7 distance: {tile_hs.get(7, 0)}")
    print(f"                                     Tile 8 distance: {tile_hs.get(8, 0)}")
    print("==========================================================\n")

def animate_solution(solution_path, goal_state, heuristic_choice, delay=3.0):
    """Loops through the path, clearing the output and rendering each frame."""
    total_steps = len(solution_path)
    for index, node in enumerate(solution_path):
        clear_output(wait=True)
        display_puzzle_frame(node, index, total_steps - 1, goal_state, heuristic_choice)
        if index < total_steps - 1:
            time.sleep(delay)
    print("Playback Finished!")

In [35]:
def solve_and_animate_8_puzzle(initial_state, goal_state, heuristic_choice):
    """Solves the puzzle and triggers the step-by-step frame animation with heuristic table."""
    # Setup initial node
    initial_node = create_node(initial_state)
    update_hn(initial_node, calculate_distance(initial_state, goal_state, heuristic_choice))

    fringe = [initial_node]
    explored_nodes = []
    count = 1

    print("Searching for a solution...")
    while fringe:
        min_fn_index = find_least_fn(fringe)
        if min_fn_index == -1:
            print("No solution found.")
            return

        current_node = fringe.pop(min_fn_index)
        current_state = get_current_state(current_node)

        if current_state == goal_state:
            full_explored = explored_nodes + [current_node]
            path = get_solution_path(full_explored)
            print(f"Solution found! Starting playback of {len(path)} frames...")
            time.sleep(2)
            animate_solution(path, goal_state, heuristic_choice, delay=1.5)
            return

        zero_index = current_state.index(0)
        g_cost = get_gn(current_node) + 1
        count = expand_node(fringe, explored_nodes, current_node, goal_state, zero_index, g_cost, count, heuristic_choice)

    print("No solution found.")

# Run the animation solver
solve_and_animate_8_puzzle(initial_state, goal_state, heuristic_choice)

                   PLAYBACK FRAME: 26 / 26
  g(n) [Path Cost]:       26
  h(n) [Heuristic Value]: 0
  f(n) [Total Cost]:      26
----------------------------------------------------------
        BOARD STATE                  TILE HEURISTIC TABLE  
        1 | 2 | 3                  Tile 1 distance: 0
       -----------                   Tile 2 distance: 0
        4 | 5 | 6                  Tile 3 distance: 0
       -----------                   Tile 4 distance: 0
        7 | 8 | 0                  Tile 5 distance: 0
                                     Tile 6 distance: 0
                                     Tile 7 distance: 0
                                     Tile 8 distance: 0

Playback Finished!
